# Exercise: Implementing a First-Come, First-Served Scheduler

Welcome to the exercise on implementing a basic scheduler! In the fast-paced world of LLM serving, the scheduler is the traffic controller, deciding which requests get to use the powerful GPU resources and when. A well-designed scheduler is key to maximizing throughput and ensuring fairness.

In this exercise, you will implement a `FirstComeFirstServedScheduler`. This is one of the simplest scheduling strategies, but understanding it is a fantastic first step toward grasping more complex systems like the one used in vLLM.

**In this exercise you will:**
*   Implement a method to add new, incoming requests to a pending queue.
*   Implement the core scheduling logic to select a batch of requests to run, based on availability.
*   Implement a method to update the status of requests once they have finished processing.

Let's get started!

In [ ]:
import collections
import dataclasses
from enum import Enum
from typing import Deque, List

# Helper code - You don't need to modify this!
# ---------------------------------------------

class RequestStatus(Enum):
    """Enumeration for the status of a request."""
    PENDING = "PENDING"
    RUNNING = "RUNNING"
    COMPLETED = "COMPLETED"

@dataclasses.dataclass
class Request:
    """Represents a single inference request."""
    request_id: str
    prompt: str
    status: RequestStatus = RequestStatus.PENDING

    # Make Request hashable so it can be used in sets
    def __hash__(self):
        return hash(self.request_id)

    def __eq__(self, other):
        if not isinstance(other, Request):
            return NotImplemented
        return self.request_id == other.request_id

# This is the class you will be implementing.
# -------------------------------------------

class FirstComeFirstServedScheduler:
    """
    A simple scheduler that processes requests in the order they are received.
    """
    def __init__(self):
        self.pending_queue: Deque[Request] = collections.deque()
        self.running_requests: List[Request] = []
        self.completed_log: List[Request] = []

### Exercise 1: Adding Requests to the Queue

Your first task is to implement the `add_requests` method. This method is called whenever new inference requests arrive at the server. Your implementation should take a list of new `Request` objects, set their status to `PENDING`, and add them to the `pending_queue`.

**Hint:** The `pending_queue` is a `collections.deque`, which has an efficient `append()` method for adding items to the end of the queue.

In [ ]:
def add_requests(self, new_requests: List[Request]) -> None:
    """
    Adds a new list of requests to the pending queue.

    Args:
        new_requests: A list of Request objects to be added.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Add the method to our class
FirstComeFirstServedScheduler.add_requests = add_requests

In [ ]:
# Test for Exercise 1: add_requests

print("Testing add_requests...")

# 1. Setup
scheduler = FirstComeFirstServedScheduler()
req1 = Request(request_id="abc", prompt="Hello")
req2 = Request(request_id="def", prompt="World")
new_requests = [req1, req2]

# 2. Action
scheduler.add_requests(new_requests)

# 3. Assertions
assert len(scheduler.pending_queue) == 2, f"Expected pending queue length of 2, but got {len(scheduler.pending_queue)}"
print("✅ Correct queue length.")

assert scheduler.pending_queue[0].request_id == "abc", "The first request was not added correctly."
assert scheduler.pending_queue[1].request_id == "def", "The second request was not added correctly."
print("✅ Requests are in the correct order.")

for req in scheduler.pending_queue:
    assert req.status == RequestStatus.PENDING, f"Request {req.request_id} should have PENDING status, but has {req.status}"
print("✅ All new requests have PENDING status.")

print("\nAll tests for add_requests passed! Great job!")

### Exercise 2: Selecting the Execution Batch

Now for the core logic! You will implement `select_batch`. This method decides which requests to run in the next inference step.

Your implementation should:
1.  Determine how many requests to schedule, which is the *minimum* of the number of pending requests and the `max_batch_size`.
2.  For that many requests:
    *   Remove the request from the *front* of the `pending_queue`.
    *   Update its status to `RUNNING`.
    *   Add it to the `self.running_requests` list.
3.  Return the list of requests selected to be in the batch.

In [ ]:
def select_batch(self, max_batch_size: int) -> List[Request]:
    """
    Selects the next batch of requests to run from the pending queue.

    Args:
        max_batch_size: The maximum number of requests to include in the batch.

    Returns:
        A list of Request objects that were selected to run.
    """
    batch_to_run = []
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###
    return batch_to_run

# Add the method to our class
FirstComeFirstServedScheduler.select_batch = select_batch

In [ ]:
# Test for Exercise 2: select_batch

print("Testing select_batch...")

# 1. Setup
scheduler = FirstComeFirstServedScheduler()
requests_to_add = [
    Request(request_id=f"req_{i}", prompt=f"p_{i}") for i in range(5)
]
scheduler.add_requests(requests_to_add)

# 2. Action
max_batch_size = 3
selected_batch = scheduler.select_batch(max_batch_size)

# 3. Assertions
assert len(selected_batch) == 3, f"Expected batch size of 3, but got {len(selected_batch)}"
print("✅ Correct batch size returned.")

assert len(scheduler.pending_queue) == 2, f"Expected 2 requests left in pending queue, but found {len(scheduler.pending_queue)}"
assert scheduler.pending_queue[0].request_id == "req_3", "The wrong requests are left in the pending queue."
print("✅ Correct requests removed from pending queue.")

assert len(scheduler.running_requests) == 3, f"Expected 3 requests in running_requests list, but found {len(scheduler.running_requests)}"
print("✅ Correct number of requests in running list.")

for req in selected_batch:
    assert req.status == RequestStatus.RUNNING, f"Request {req.request_id} in batch should have RUNNING status, but has {req.status}"
print("✅ Request statuses correctly updated to RUNNING.")

# Test case with fewer pending requests than max_batch_size
scheduler = FirstComeFirstServedScheduler()
scheduler.add_requests([Request(request_id="last_req", prompt="p")])
last_batch = scheduler.select_batch(max_batch_size=5)
assert len(last_batch) == 1, f"Expected batch size of 1, but got {len(last_batch)}"
print("✅ Handles cases with fewer pending requests than max_batch_size.")


print("\nAll tests for select_batch passed! Fantastic work!")

### Exercise 3: Handling Completed Requests

You're almost there! The final step is to handle batches that have finished processing.

Implement the `on_batch_completed` method. This function is called by the system after the model has run inference on a batch.

Your implementation should:
1.  Iterate through the `finished_batch` provided.
2.  For each request, update its status to `COMPLETED`.
3.  Append these completed requests to the `self.completed_log`.
4.  Remove the completed requests from the `self.running_requests` list.

In [ ]:
def on_batch_completed(self, finished_batch: List[Request]) -> None:
    """
    Updates the state of the scheduler after a batch has finished processing.

    Args:
        finished_batch: The list of Request objects that have completed.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Add the method to our class
FirstComeFirstServedScheduler.on_batch_completed = on_batch_completed

In [ ]:
# Test for Exercise 3: on_batch_completed

print("Testing on_batch_completed...")

# 1. Setup
scheduler = FirstComeFirstServedScheduler()
requests_to_add = [
    Request(request_id=f"req_{i}", prompt=f"p_{i}") for i in range(5)
]
scheduler.add_requests(requests_to_add)

# Simulate a batch being selected and run
batch1 = scheduler.select_batch(max_batch_size=3)
# Let's add another request that is still running and should not be affected
still_running_req = Request(request_id="still_running", prompt="...")
still_running_req.status = RequestStatus.RUNNING
scheduler.running_requests.append(still_running_req)


# 2. Action
scheduler.on_batch_completed(batch1)

# 3. Assertions
assert len(scheduler.completed_log) == 3, f"Expected completed_log to have 3 requests, but found {len(scheduler.completed_log)}"
print("✅ Completed requests added to log.")

for req in batch1:
    assert req.status == RequestStatus.COMPLETED, f"Request {req.request_id} should have status COMPLETED, but has {req.status}"
print("✅ Request statuses correctly updated to COMPLETED.")

assert len(scheduler.running_requests) == 1, f"Expected 1 request remaining in running_requests, but found {len(scheduler.running_requests)}"
assert scheduler.running_requests[0].request_id == "still_running", "The wrong request was removed from running_requests."
print("✅ Correctly removed completed requests from the running list.")


print("\nAll tests for on_batch_completed passed! You've built a complete scheduler!")

### Challenge (Optional)

Congratulations on implementing a fully functional FCFS scheduler!

For an extra challenge, think about how you could improve `select_batch`. In a real system like vLLM, the scheduler doesn't just count requests; it counts *tokens*. A batch might be limited to 32 requests OR a total of 20,000 tokens, whichever comes first.

**Challenge:**
- Modify the `Request` dataclass to include a `token_count: int`.
- Modify your `select_batch` implementation to accept a `max_total_tokens` argument.
- The new logic should stop adding requests to the batch if either `max_batch_size` or `max_total_tokens` is exceeded.

This is purely optional, but it's a great way to think about the real-world constraints that advanced schedulers have to manage.

In [ ]:
# Integration Test: Simulating the full lifecycle

print("--- Running Full Scheduler Simulation ---")

# Setup
scheduler = FirstComeFirstServedScheduler()
max_batch_size = 2

# Step 1: Five requests arrive
print("\nStep 1: 5 new requests arrive.")
initial_requests = [Request(request_id=f"req_{i}", prompt="...") for i in range(5)]
scheduler.add_requests(initial_requests)
assert len(scheduler.pending_queue) == 5
print("State: Pending=5, Running=0, Completed=0")

# Step 2: First scheduling step
print("\nStep 2: Scheduler selects the first batch.")
batch1 = scheduler.select_batch(max_batch_size)
assert len(batch1) == 2
assert batch1[0].request_id == "req_0" and batch1[1].request_id == "req_1"
assert len(scheduler.pending_queue) == 3
assert len(scheduler.running_requests) == 2
print("State: Pending=3, Running=2, Completed=0")

# Step 3: Two more requests arrive while batch 1 is processing
print("\nStep 3: 2 more requests arrive.")
more_requests = [Request(request_id=f"req_{i}", prompt="...") for i in range(5, 7)]
scheduler.add_requests(more_requests)
assert len(scheduler.pending_queue) == 5 # (req_2, req_3, req_4, req_5, req_6)
print("State: Pending=5, Running=2, Completed=0")

# Step 4: Batch 1 finishes
print("\nStep 4: Batch 1 completes processing.")
scheduler.on_batch_completed(batch1)
assert len(scheduler.running_requests) == 0
assert len(scheduler.completed_log) == 2
print("State: Pending=5, Running=0, Completed=2")

# Step 5: Second scheduling step
print("\nStep 5: Scheduler selects the second batch.")
batch2 = scheduler.select_batch(max_batch_size)
assert len(batch2) == 2
assert batch2[0].request_id == "req_2" and batch2[1].request_id == "req_3"
assert len(scheduler.pending_queue) == 3
assert len(scheduler.running_requests) == 2
print("State: Pending=3, Running=2, Completed=2")

# Step 6: Batch 2 finishes
print("\nStep 6: Batch 2 completes processing.")
scheduler.on_batch_completed(batch2)
assert len(scheduler.running_requests) == 0
assert len(scheduler.completed_log) == 4
print("State: Pending=3, Running=0, Completed=4")

print("\n--- Simulation Complete ---")
print("Congratulations! Your scheduler correctly handled the entire request lifecycle.")